In [ ]:
# Cell 1 — Install libraries
!pip install -q torch pandas numpy scikit-learn
print('Libraries ready')


In [ ]:
# Cell 2 — Configuration
import torch
import numpy as np
import pandas as pd

torch.manual_seed(42)
np.random.seed(42)

# Sequence / architecture config
SEQUENCE_LENGTH = 20
TCN_CHANNELS    = 32      # reduced from 64 — better fit for ~1,800 sequences
DROPOUT         = 0.3     # increased from 0.2 — more regularisation for larger dataset
LEARNING_RATE   = 3e-5
WEIGHT_DECAY    = 1e-5
HUBER_DELTA     = 0.5
BATCH_SIZE      = 32
MAX_EPOCHS      = 250
PATIENCE        = 30
VAL_FRACTION    = 0.15    # last 15% of training data used as validation (chronological)

print('Config set')
print(f'  TCN_CHANNELS = {TCN_CHANNELS}')
print(f'  DROPOUT      = {DROPOUT}')
print(f'  LEARNING_RATE= {LEARNING_RATE}')


In [ ]:
# Cell 3 — Upload all 5 input files
from google.colab import files
import pandas as pd

print('Upload nifty50_step4_standard_train.csv (numerical train)')
up = files.upload()
train_df = pd.read_csv(list(up.keys())[0])

print('\nUpload nifty50_step4_standard_test.csv (numerical test)')
up = files.upload()
test_df = pd.read_csv(list(up.keys())[0])

print('\nUpload step7b_finbert_embeddings.csv (768-dim embeddings)')
up = files.upload()
emb_df = pd.read_csv(list(up.keys())[0])

print('\nUpload step7a_finbert_sentiment_comparison.csv (sentiment scores)')
up = files.upload()
sent_df = pd.read_csv(list(up.keys())[0])

print('\nUpload standard_scaler.pkl (optional — for reference only, not required for directional accuracy)')
up = files.upload()
scaler_filename = list(up.keys())[0] if up else None

print()
print(f'Train  : {train_df.shape}')
print(f'Test   : {test_df.shape}')
print(f'Embed  : {emb_df.shape}')
print(f'Sentim : {sent_df.shape}')


In [ ]:
# Cell 4 — Merge numerical + FinBERT embeddings + sentiment scores
import pandas as pd
import numpy as np

# Parse dates — all files use dd-Mon-YYYY format
train_df['date_p'] = pd.to_datetime(train_df['Date'], format='%d-%b-%Y', errors='coerce')
test_df['date_p']  = pd.to_datetime(test_df['Date'],  format='%d-%b-%Y', errors='coerce')
emb_df['date_p']   = pd.to_datetime(emb_df['date'],   format='%d-%b-%Y', errors='coerce')
sent_df['date_p']  = pd.to_datetime(sent_df['date'],  format='%d-%b-%Y', errors='coerce')

emb_cols  = [c for c in emb_df.columns if c.startswith('emb_')]
sent_cols = ['finbert_score', 'gpt4o_score', 'finbert_pos_prob', 'finbert_neg_prob', 'finbert_neu_prob']

# CRITICAL FIX: merge BOTH the 768-dim embeddings AND the 5 sentiment score
# features together. In earlier runs only the embeddings file was used,
# which silently dropped the sentiment scores (finbert_score, gpt4o_score, etc.)
# — these carry a strong, statistically significant correlation with next-day
# direction (r ~ 0.3-0.4, p < 0.001) and their absence likely caused the
# accuracy collapse seen on the larger 9-year dataset.
text_df = emb_df[['date_p'] + emb_cols].merge(
    sent_df[['date_p'] + sent_cols], on='date_p', how='outer'
)

def merge_text(num_df, label):
    m = num_df.merge(text_df, on='date_p', how='left')
    coverage = m[emb_cols[0]].notna().mean()
    m[emb_cols + sent_cols] = m[emb_cols + sent_cols].fillna(0.0)
    print(f'{label} embedding coverage: {coverage*100:.1f}%  ({len(m)} rows)')
    return m

train_m = merge_text(train_df, 'Train')
test_m  = merge_text(test_df,  'Test')

EXCLUDE_COLS = ['Date', 'date', 'date_p', 'Close', 'LogReturn_Close'] + emb_cols + sent_cols
NUM_FEATURE_COLS = [c for c in train_m.columns if c not in EXCLUDE_COLS]
TEXT_COLS = emb_cols + sent_cols
TARGET_COL = 'LogReturn_Close'

print()
print(f'Numerical features ({len(NUM_FEATURE_COLS)}): {NUM_FEATURE_COLS}')
print(f'Text vector dim: {len(TEXT_COLS)}  ({len(emb_cols)} embedding dims + {len(sent_cols)} sentiment scores)')


In [ ]:
# Cell 5 — Build sliding window sequences
import numpy as np

def build_sequences(df, seq_len):
    df = df.sort_values('date_p').reset_index(drop=True)
    num    = df[NUM_FEATURE_COLS].values.astype(np.float32)
    text   = df[TEXT_COLS].values.astype(np.float32)
    target = df[TARGET_COL].values.astype(np.float32)

    X_num, X_txt, y = [], [], []
    for i in range(seq_len, len(df)):
        X_num.append(num[i-seq_len:i])    # past seq_len days of indicators
        X_txt.append(text[i])             # text features of the prediction day
        y.append(target[i])               # log return to predict
    return np.array(X_num), np.array(X_txt), np.array(y)

Xn_train, Xt_train, y_train = build_sequences(train_m, SEQUENCE_LENGTH)
Xn_test,  Xt_test,  y_test  = build_sequences(test_m,  SEQUENCE_LENGTH)

print(f'Train sequences : {Xn_train.shape}  text: {Xt_train.shape}  target: {y_train.shape}')
print(f'Test  sequences : {Xn_test.shape}   text: {Xt_test.shape}   target: {y_test.shape}')
print()
print(f'Train UP ratio  : {(y_train > 0).mean()*100:.2f}%')
print(f'Test  UP ratio  : {(y_test  > 0).mean()*100:.2f}%')


In [ ]:
# Cell 6 — Model definition: TCN + Deep Fusion FNN
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class SeqDataset(Dataset):
    def __init__(self, Xn, Xt, y):
        self.Xn = torch.tensor(Xn, dtype=torch.float32)
        self.Xt = torch.tensor(Xt, dtype=torch.float32)
        self.y  = torch.tensor(y,  dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.Xn[idx], self.Xt[idx], self.y[idx]


class TCNBlock(nn.Module):
    """Dilated causal 1D conv block with residual connection."""
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.pad  = pad
        self.res  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out = out[:, :, :-self.pad] if self.pad > 0 else out   # causal trim
        out = self.act(self.bn(out))
        out = self.drop(out)
        return out + self.res(x)


class TCNDeepFNN(nn.Module):
    """
    Numerical branch : 4 stacked dilated TCN blocks (dilations 1,2,4,8)
                        capture 1/2/4/8-day momentum patterns.
    Textual branch   : Linear projection of FinBERT embeddings + sentiment
                        scores (773-dim).
    Fusion           : Concatenate -> deep FNN with residual-friendly
                        progressive layer sizes -> single output (log return).
    """
    def __init__(self, num_features, text_dim, tcn_channels=32, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(num_features, tcn_channels)
        self.tcn1 = TCNBlock(tcn_channels, tcn_channels, dilation=1, dropout=dropout)
        self.tcn2 = TCNBlock(tcn_channels, tcn_channels, dilation=2, dropout=dropout)
        self.tcn3 = TCNBlock(tcn_channels, tcn_channels, dilation=4, dropout=dropout)
        self.tcn4 = TCNBlock(tcn_channels, tcn_channels, dilation=8, dropout=dropout)

        self.txt_proj = nn.Sequential(
            nn.Linear(text_dim, tcn_channels),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        fusion_in = tcn_channels * 2
        self.fnn = nn.Sequential(
            nn.Linear(fusion_in, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 32),         nn.GELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_num, x_txt):
        x = self.proj(x_num)          # (B, L, C)
        x = x.transpose(1, 2)         # (B, C, L)
        x = self.tcn1(x); x = self.tcn2(x); x = self.tcn3(x); x = self.tcn4(x)
        x = x[:, :, -1]                # last timestep -> (B, C)

        t = self.txt_proj(x_txt)       # (B, C)

        fused = torch.cat([x, t], dim=1)
        return self.fnn(fused).squeeze(-1)


print('Model classes defined')


In [ ]:
# Cell 7 — Training loop
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

train_ds_full = SeqDataset(Xn_train, Xt_train, y_train)
test_ds       = SeqDataset(Xn_test,  Xt_test,  y_test)

# Chronological train/validation split — last VAL_FRACTION of training data
n_val = int(len(train_ds_full) * VAL_FRACTION)
n_tr  = len(train_ds_full) - n_val
train_ds = torch.utils.data.Subset(train_ds_full, range(0, n_tr))
val_ds   = torch.utils.data.Subset(train_ds_full, range(n_tr, len(train_ds_full)))

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

model = TCNDeepFNN(
    num_features=Xn_train.shape[2],
    text_dim=Xt_train.shape[1],
    tcn_channels=TCN_CHANNELS,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.HuberLoss(delta=HUBER_DELTA)

print()
print('=' * 60)
print('  TRAINING — TCN + Deep Fusion FNN (9-year dataset)')
print('=' * 60)
print(f'  Loss      : Huber (delta={HUBER_DELTA})')
print(f'  Optimiser : Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})')
print(f'  Channels  : {TCN_CHANNELS}   Dropout: {DROPOUT}')
print(f'  Patience  : {PATIENCE} epochs')
print()

best_val_loss = float('inf')
best_state = None
best_epoch = 0
no_improve = 0
train_losses, val_losses = [], []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_train_loss = []
    for xn, xt, y in train_loader:
        xn, xt, y = xn.to(device), xt.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(xn, xt)
        loss = loss_fn(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        epoch_train_loss.append(loss.item())

    model.eval()
    epoch_val_loss = []
    with torch.no_grad():
        for xn, xt, y in val_loader:
            xn, xt, y = xn.to(device), xt.to(device), y.to(device)
            pred = model(xn, xt)
            epoch_val_loss.append(loss_fn(pred, y).item())

    tr_loss  = np.mean(epoch_train_loss)
    val_loss = np.mean(epoch_val_loss)
    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{MAX_EPOCHS}  Train={tr_loss:.4f}  Val={val_loss:.4f}'
              f'  (best epoch {best_epoch}, val={best_val_loss:.4f})')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print()
print(f'Best epoch    : {best_epoch}')
print(f'Best val loss : {best_val_loss:.4f}')
print('Best weights loaded for evaluation')


In [ ]:
# Cell 8 — Evaluation on test set
import numpy as np

model.eval()
preds, actuals = [], []
with torch.no_grad():
    for xn, xt, y in test_loader:
        xn, xt = xn.to(device), xt.to(device)
        pred = model(xn, xt)
        preds.extend(pred.cpu().numpy())
        actuals.extend(y.numpy())

preds   = np.array(preds)
actuals = np.array(actuals)

mae  = np.mean(np.abs(preds - actuals))
rmse = np.sqrt(np.mean((preds - actuals) ** 2))
ss_res = np.sum((actuals - preds) ** 2)
ss_tot = np.sum((actuals - actuals.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

dir_acc = ((preds > 0) == (actuals > 0)).mean()

# MAPE — guarded against near-zero actuals
mask = np.abs(actuals) > 1e-4
mape = np.mean(np.abs((actuals[mask] - preds[mask]) / actuals[mask])) * 100 if mask.sum() > 0 else float('nan')

print('=' * 65)
print('  EVALUATION — TCN + Deep Fusion FNN (9-year dataset, retuned)')
print('=' * 65)
print(f'  MAE                  : {mae:.6f}')
print(f'  RMSE                 : {rmse:.6f}')
print(f'  MAPE                 : {mape:.2f}%')
print(f'  R²                   : {r2:.4f}')
print(f'  Directional Accuracy : {dir_acc*100:.2f}%')
print(f'  Best epoch           : {best_epoch}')
print()

print('-' * 50)
print('  COMPARISON TABLE')
print('-' * 50)
print(f'  Naive baseline                       : ~51.00%')
print(f'  Baseline LSTM (OHLC only, 11yr)      :  50.94%')
print(f'  TCN + Deep FNN (3yr data)             :  58.09%')
print(f'  TCN + Deep FNN (9yr data, this run)   :  {dir_acc*100:5.2f}%')
print()

if dir_acc > 0.5809:
    print(f'  ✅ Improved over previous best by +{(dir_acc - 0.5809)*100:.2f} percentage points')
else:
    print(f'  ⚠️  Did not exceed previous best (58.09%)')


In [ ]:
# Cell 9 — Save predictions, metrics, training curve and model weights
import pandas as pd
import numpy as np
import torch

# Predictions vs actuals, aligned to test dates (offset by SEQUENCE_LENGTH)
test_dates = test_m.sort_values('date_p').reset_index(drop=True)['date_p'].iloc[SEQUENCE_LENGTH:].reset_index(drop=True)

pred_df = pd.DataFrame({
    'date': test_dates.dt.strftime('%d-%b-%Y'),
    'predicted_log_return': preds,
    'actual_log_return': actuals,
    'predicted_direction': np.where(preds > 0, 'UP', 'DOWN'),
    'actual_direction': np.where(actuals > 0, 'UP', 'DOWN'),
    'correct': (preds > 0) == (actuals > 0)
})
pred_df.to_csv('step9_predictions.csv', index=False)

metrics_df = pd.DataFrame([{
    'MAE': mae,
    'RMSE': rmse,
    'MAPE_pct': mape,
    'R2': r2,
    'Directional_Accuracy_pct': dir_acc * 100,
    'Best_Epoch': best_epoch,
    'TCN_Channels': TCN_CHANNELS,
    'Dropout': DROPOUT,
    'Learning_Rate': LEARNING_RATE,
    'Sequence_Length': SEQUENCE_LENGTH,
    'Train_Sequences': len(Xn_train),
    'Test_Sequences': len(Xn_test),
}])
metrics_df.to_csv('step9_evaluation_metrics.csv', index=False)

loss_df = pd.DataFrame({
    'epoch': range(1, len(train_losses) + 1),
    'train_loss': train_losses,
    'val_loss': val_losses
})
loss_df.to_csv('step9_training_loss.csv', index=False)

torch.save(best_state, 'step9_model_weights.pth')

print('=' * 60)
print('  STEP 9 COMPLETE — ALL OUTPUT FILES')
print('=' * 60)
print('  ✓ step9_predictions.csv')
print('  ✓ step9_evaluation_metrics.csv')
print('  ✓ step9_training_loss.csv')
print('  ✓ step9_model_weights.pth')
print()
print(f'  Directional Accuracy : {dir_acc*100:.2f}%')
print(f'  R²                   : {r2:.4f}')
print(f'  Best epoch           : {best_epoch}')
print()

from google.colab import files
for f in ['step9_predictions.csv', 'step9_evaluation_metrics.csv',
          'step9_training_loss.csv', 'step9_model_weights.pth']:
    files.download(f)

print('✅ Done — use these files in your thesis results chapter')
